# Import

In [ ]:
!pip install torch_geometric

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import pickle

import torch
from torch_geometric.data import Data

import gc
from collections import defaultdict

# Change directory

In [ ]:
# Cambiar al directorio
os.chdir('/content/drive/MyDrive/nids-mitre/')

# Verificar que estás ahí
!pwd


/content/drive/MyDrive/nids-mitre


# Explore data

In [ ]:
!unzip /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964.zip \
  "*/data/NF-CICIDS2018-v3.csv" \
  -d /content/drive/MyDrive/nids-mitre/data/cicids2018-v3


In [ ]:
!head /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv

In [ ]:
!awk -F',' 'NR>1 {print $3; print $5}' \
  /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv \
  | sort -u | wc -l

205801


In [ ]:
# ============================================================================
# ANÁLISIS EXPLORATORIO SIN CARGAR TODO EN MEMORIA
# ============================================================================

# 1. Leer SOLO las primeras filas para ver estructura
df_sample = pd.read_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv', nrows=10000)

print("Columnas disponibles:")
print(df_sample.columns.tolist())
print(f"\nShape muestra: {df_sample.shape}")
print(f"\nTipos de datos:")
print(df_sample.dtypes)

# 2. Contar filas TOTALES sin cargar todo
total_rows = sum(1 for _ in open('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv')) - 1  # -1 por header
print(f"\n✓ Total de flows: {total_rows:,}")

# 3. Identificar columnas clave
timestamp_col_start = 'FLOW_START_MILLISECONDS'
timestamp_col_end = 'FLOW_END_MILLISECONDS'
src_ip_col = 'IPV4_SRC_ADDR'
dst_ip_col = 'IPV4_DST_ADDR'
label_col = 'Label'

print(f"\nColumnas identificadas:")
print(f"  Timestamp: {timestamp_col_start}")
print(f"  Timestamp: {timestamp_col_end}")
print(f"  Source IP: {src_ip_col}")
print(f"  Dest IP: {dst_ip_col}")
print(f"  Label: {label_col}")

# 4. Ver distribución de labels (sin cargar todo)
print("\nDistribución de labels (muestra):")
print(df_sample[label_col].value_counts())

# 5. Ver rango temporal
df_sample[timestamp_col_start] = pd.to_datetime(df_sample[timestamp_col_start])
print(f"\nRango temporal (muestra):")
print(f"  Inicio: {df_sample[timestamp_col_start].min()}")
print(f"  Fin: {df_sample[timestamp_col_start].max()}")

Columnas disponibles:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SR

# Filter and save chunks

In [ ]:
selected_ips_list = pd.read_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/ip_statistics.csv')
# NOTE: I used all ips, not only the most relevant (!!)

In [ ]:
# ============================================================================
# GUARDAR CHUNKS FILTRADOS INCREMENTALMENTE (RECOMENDADA)
# ============================================================================

def filter_and_save_incrementally(csv_path, selected_ips, timestamp_col,
                                  src_ip_col, dst_ip_col, label_col,
                                  output_file,
                                  chunk_size):
    """
    Filtra y guarda datos incrementalmente SIN concatenar en memoria
    """

    total_rows_kept = 0
    chunk_count = 0

    print("Filtrando y guardando por chunks...")
    print(f"Archivo de salida: {output_file}")

    # Abrir archivo de salida en modo append binario
    with open(output_file, 'wb') as f_out:

        for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
            print(i, len(chunk))
            print(f"  Procesando chunk {i+1}... ({total_rows_kept:,} rows guardadas)", end='\r')

            # Filtrar
            mask = (
                chunk[src_ip_col].isin(selected_ips['ip']) |
                chunk[dst_ip_col].isin(selected_ips['ip'])
            )

            filtered = chunk[mask].copy()

            if len(filtered) > 0:
                # Convertir timestamp
                filtered[timestamp_col] = pd.to_datetime(filtered[timestamp_col])

                # Guardar este chunk
                pickle.dump(filtered, f_out)

                total_rows_kept += len(filtered)
                chunk_count += 1

            # Liberar memoria
            del chunk, filtered, mask

    print(f"\n✓ Filtrado completado:")
    print(f"  Total rows: {total_rows_kept:,}")
    print(f"  Chunks guardados: {chunk_count}")
    print(f"  Tamaño archivo: {os.path.getsize(output_file) / (1024**2):.2f} MB")

    return total_rows_kept, chunk_count

# Ejecutar filtrado
total_rows, num_chunks = filter_and_save_incrementally(
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv',
    selected_ips_list,
    timestamp_col_start,
    src_ip_col,
    dst_ip_col,
    label_col,
    output_file='/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl',
    chunk_size=250000
)

Filtrando y guardando por chunks...
Archivo de salida: /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl
0 250000
1 250000
2 250000
3 250000
4 250000
5 250000
6 250000
7 250000
8 250000
9 250000
10 250000
11 250000
12 250000
13 250000
14 250000
15 250000
16 250000
17 250000
18 250000
19 250000
20 250000
21 250000
22 250000
23 250000
24 250000
25 250000
26 250000
27 250000
28 250000
29 250000
30 250000
31 250000
32 250000
33 250000
34 250000
35 250000
36 250000
37 250000
38 250000
39 250000
40 250000
41 250000
42 250000
43 250000
44 250000
45 250000
46 250000
47 250000
48 250000
49 250000
50 250000
51 250000
52 250000
53 250000
54 250000
55 250000
56 250000
57 250000
58 250000
59 250000
60 250000
61 250000
62 250000
63 250000
64 250000
65 250000
66 250000
67 250000
68 250000
69 250000
70 250000
71 250000
72 250000
73 250000
74 250000
75 250000
76 250000
77 250000
78 250000
79 250000
80 115529

✓ Filtrado completado:
  Total rows: 20,115,529
  Chunks guardados: 81


# Read chunks

In [ ]:
# ============================================================================
# LEER CHUNKS GUARDADOS UNO A LA VEZ
# ============================================================================

def read_filtered_chunks(pickle_file):
    """
    Generador que lee chunks de uno en uno del archivo pickle
    """

    with open(pickle_file, 'rb') as f:
        while True:
            try:
                chunk = pickle.load(f)
                yield chunk
            except EOFError:
                break

# Ejemplo de uso:
print("\nProbando lectura de chunks...")
for i, chunk in enumerate(read_filtered_chunks('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl')):
    print(f"Chunk {i+1}: {chunk.shape}")
    if i >= 2:  # Solo mostrar primeros 3 chunks
        print("...")
        break


Probando lectura de chunks...
Chunk 1: (250000, 55)
Chunk 2: (250000, 55)
Chunk 3: (250000, 55)
...


# Create graphs

In [ ]:

# ============================================================================
# VERSIÓN ULTRA-OPTIMIZADA: Una sola pasada por los chunks
# ============================================================================

def create_graph_fixed_nodes_CORRECTED(window_df, src_col, dst_col, label_col,
                                       global_ip_to_idx, num_total_nodes):
    """
    Crea un grafo con conjunto FIJO de nodos

    Args:
        window_df: DataFrame con flows de una ventana temporal
        src_col: Columna de IP origen
        dst_col: Columna de IP destino
        label_col: Columna de etiqueta
        global_ip_to_idx: Mapeo GLOBAL IP → índice (fijo)
        num_total_nodes: Número total de nodos (fijo)

    Returns:
        Data: Objeto PyTorch Geometric con grafo

    Versión CORREGIDA con labels numéricos

    Label encoding:
    - 0 = Benign
    - 1+ = Ataque (diferentes tipos)
    """

    from collections import defaultdict
    import torch
    from torch_geometric.data import Data
    import numpy as np
    import pandas as pd

    # ========================================================================
    # 1. AGREGAR FLOWS POR PAR DE IPs (evitar multi-edges)
    # ========================================================================

    edge_flows = defaultdict(list)

    for _, row in window_df.iterrows():
        src_idx = global_ip_to_idx[row[src_col]]
        dst_idx = global_ip_to_idx[row[dst_col]]

        edge_key = (src_idx, dst_idx)
        edge_flows[edge_key].append(row)

    edge_list = []
    edge_features = []

    feature_cols = [
        'FLOW_DURATION_MILLISECONDS',
        'IN_BYTES',
        'IN_PKTS',
        'OUT_BYTES',
        'OUT_PKTS'
    ]

    available_cols = [c for c in feature_cols if c in window_df.columns]

    for (src_idx, dst_idx), flows in edge_flows.items():

        edge_list.append([src_idx, dst_idx])

        flows_df = pd.DataFrame(flows)

        agg_features = []

        for col in available_cols:
            agg_features.append(float(flows_df[col].sum()))
            agg_features.append(float(flows_df[col].mean()))
            agg_features.append(float(flows_df[col].std() if len(flows_df) > 1 else 0.0))

        agg_features.append(float(len(flows)))

        edge_features.append(agg_features)

    # ========================================================================
    # 2. NODE FEATURES
    # ========================================================================

    node_activity = defaultdict(list)

    for _, row in window_df.iterrows():
        src_ip = row[src_col]
        dst_ip = row[dst_col]

        node_activity[src_ip].append({
            'role': 'source',
            'bytes': row.get('IN_BYTES', 0),
            'pkts': row.get('IN_PKTS', 0),
            'duration': row.get('FLOW_DURATION_MILLISECONDS', 0)
        })

        node_activity[dst_ip].append({
            'role': 'destination',
            'bytes': row.get('OUT_BYTES', 0),
            'pkts': row.get('OUT_PKTS', 0),
            'duration': row.get('FLOW_DURATION_MILLISECONDS', 0)
        })

    node_features = []

    for ip in sorted(global_ip_to_idx.keys(), key=lambda x: global_ip_to_idx[x]):

        if ip in node_activity:
            activities = node_activity[ip]

            total_flows = len(activities)
            total_bytes = sum(a['bytes'] for a in activities)
            total_pkts = sum(a['pkts'] for a in activities)
            avg_duration = np.mean([a['duration'] for a in activities])
            std_duration = np.std([a['duration'] for a in activities])

            as_source = sum(1 for a in activities if a['role'] == 'source')
            as_dest = sum(1 for a in activities if a['role'] == 'destination')

            features = [
                float(total_flows),
                float(as_source),
                float(as_dest),
                float(total_bytes),
                float(total_pkts),
                float(avg_duration),
                float(std_duration),
                1.0  # Nodo activo
            ]
        else:
            features = [0.0] * 8

        node_features.append(features)

    # ========================================================================
    # 3. LABEL DEL GRAFO (CORREGIDO: Labels numéricos)
    # ========================================================================

    # ✅ CORRECCIÓN: Label es numérico (0 = Benign, 1+ = Ataque)
    has_attack = (window_df[label_col] != 0).any()

    y = torch.tensor([1 if has_attack else 0], dtype=torch.long)

    # Estadísticas
    attack_flows = (window_df[label_col] != 0).sum()
    attack_ratio = attack_flows / len(window_df)

    # Print detallado
    if has_attack:
        # Tipos de ataque únicos presentes
        attack_types = window_df[window_df[label_col] != 0][label_col].unique()
        print(f"    Label: {y.item()} (ATAQUE) | "
              f"Flows ataque: {attack_flows}/{len(window_df)} ({attack_ratio*100:.1f}%) | "
              f"Tipos únicos: {len(attack_types)} | "
              f"Aristas: {len(edge_list)}")
    else:
        print(f"    Label: {y.item()} (NORMAL) | "
              f"Flows: {len(window_df)} | "
              f"Aristas: {len(edge_list)}")

    # ========================================================================
    # 4. CREAR OBJETO DATA
    # ========================================================================

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_features, dtype=torch.float)
    x = torch.tensor(node_features, dtype=torch.float)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        num_nodes=num_total_nodes
    )

    # Metadata adicional
    data.num_active_nodes = len(node_activity)
    data.num_edges_actual = len(edge_list)
    data.num_flows_original = len(window_df)
    data.attack_flows = int(attack_flows)
    data.attack_ratio = float(attack_ratio)

    return data


def create_graphs_OPTIMIZED(pickle_file, timestamp_col, src_ip_col,
                           dst_ip_col, label_col,
                           window_minutes=10,
                           output_dir='graph_data_OPTIMIZED'):
    """
    OPTIMIZACIÓN CLAVE: Lee cada chunk UNA SOLA VEZ

    Estrategia:
    1. Lee chunk
    2. Asigna flows a ventanas
    3. Acumula en buffer
    4. Cuando ventana está completa → crea grafo
    """

    import time
    import gc
    import pickle
    import os
    from collections import defaultdict

    os.makedirs(output_dir, exist_ok=True)

    print("="*70)
    print(" ⚡ VERSIÓN ULTRA-OPTIMIZADA")
    print("="*70)

    start_total = time.time()

    # ========================================================================
    # REUTILIZAR mapeo de IPs
    # ========================================================================

    #mapping_file = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/ip_to_idx_mapping.pkl'
    mapping_file = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_fixed_nodes_label0/ip_to_idx_mapping.pkl'

    print("\n✅ Cargando mapeo de IPs...")
    with open(mapping_file, 'rb') as f:
        global_ip_to_idx = pickle.load(f)

    num_nodes = len(global_ip_to_idx)
    print(f"   Nodos: {num_nodes:,}")

    # ========================================================================
    # PASADA ÚNICA: Leer chunks UNA VEZ y asignar a ventanas
    # ========================================================================

    print("\n⚡ PASADA ÚNICA: Procesando chunks...\n")

    # Buffer: {window_id: [flows]}
    window_buffer = defaultdict(list)

    # Para calcular window_id necesitamos start_time
    dataset_start = None
    dataset_end = None

    chunk_count = 0
    total_flows = 0

    for chunk in read_filtered_chunks(pickle_file):
        chunk_count += 1

        # Corregir timestamps
        ns_vals = chunk[timestamp_col].values.astype('int64')
        chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')

        # Determinar dataset_start
        if dataset_start is None:
            dataset_start = chunk[timestamp_col].min()
            dataset_end = chunk[timestamp_col].max()
        else:
            chunk_start = chunk[timestamp_col].min()
            chunk_end = chunk[timestamp_col].max()

            if chunk_start < dataset_start:
                dataset_start = chunk_start
            if chunk_end > dataset_end:
                dataset_end = chunk_end

        # Asignar window_id a cada flow
        window_size = pd.Timedelta(minutes=window_minutes)
        chunk['window_id'] = ((chunk[timestamp_col] - dataset_start) / window_size).astype(int)

        # Agrupar por ventana
        for window_id in chunk['window_id'].unique():
            window_flows = chunk[chunk['window_id'] == window_id]
            window_buffer[window_id].append(window_flows)

        total_flows += len(chunk)

        elapsed = time.time() - start_total
        print(f"  Chunk {chunk_count}/81 | "
              f"Flows: {total_flows:,} | "
              f"Ventanas en buffer: {len(window_buffer)} | "
              f"Tiempo: {elapsed:.1f}s")

        del chunk

        # Garbage collection cada 10 chunks
        if chunk_count % 10 == 0:
            gc.collect()

    total_windows = len(window_buffer)
    duration = dataset_end - dataset_start

    elapsed_pass1 = time.time() - start_total

    print(f"\n✅ Chunks procesados en {elapsed_pass1:.1f}s ({elapsed_pass1/60:.1f} min)")
    print(f"   Total flows: {total_flows:,}")
    print(f"   Rango: {dataset_start} → {dataset_end}")
    print(f"   Duración: {duration}")
    print(f"   Ventanas: {total_windows}")

    # ========================================================================
    # CREAR GRAFOS desde buffer
    # ========================================================================

    print(f"\n⚡ Creando {total_windows} grafos desde buffer...\n")

    graphs_created = 0
    graphs_with_attacks = 0
    windows_skipped = 0

    graphs_file = f'{output_dir}/graphs_optimized.pkl'

    with open(graphs_file, 'wb') as f_out:

        for window_id in sorted(window_buffer.keys()):

            if graphs_created % 100 == 0:
                elapsed = time.time() - start_total
                percent = (graphs_created / total_windows) * 100
                print(f"  Ventana {window_id} | "
                      f"Grafos: {graphs_created}/{total_windows} ({percent:.1f}%) | "
                      f"Ataques: {graphs_with_attacks} | "
                      f"Tiempo total: {elapsed:.1f}s ({elapsed/60:.1f} min)")

            # Concatenar flows de esta ventana
            window_df = pd.concat(window_buffer[window_id], ignore_index=True)

            if len(window_df) < 5:
                windows_skipped += 1
                del window_df
                continue

            # Crear grafo
            graph = create_graph_fixed_nodes_CORRECTED(
                window_df,
                src_ip_col,
                dst_ip_col,
                label_col,
                global_ip_to_idx,
                num_nodes
            )

            if graph is not None:
                graph.window_id = window_id

                # Calcular timestamp de la ventana
                window_start = dataset_start + window_id * window_size
                window_end = window_start + window_size
                graph.window_start = window_start
                graph.window_end = window_end

                pickle.dump(graph, f_out)
                graphs_created += 1

                if graph.y.item() == 1:
                    graphs_with_attacks += 1
            else:
                windows_skipped += 1

            # Liberar memoria de esta ventana
            del window_buffer[window_id]
            del window_df

            if graphs_created % 50 == 0:
                gc.collect()

    elapsed_total = time.time() - start_total

    print(f"\n\n{'='*70}")
    print("✅ PROCESAMIENTO COMPLETADO")
    print(f"{'='*70}")
    print(f"  Tiempo total: {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
    print(f"  Ventanas procesadas: {total_windows}")
    print(f"  Grafos creados: {graphs_created}")
    print(f"    Normal: {graphs_created - graphs_with_attacks}")
    print(f"    Ataque: {graphs_with_attacks} ✅")
    print(f"  Ventanas omitidas: {windows_skipped}")
    print(f"\n  Archivo: {graphs_file}")
    print(f"  Tamaño: {os.path.getsize(graphs_file) / (1024**2):.2f} MB")

    # Guardar metadata
    metadata = {
        'num_graphs': graphs_created,
        'num_attacks': graphs_with_attacks,
        'num_nodes': num_nodes,
        'window_minutes': window_minutes,
        'dataset_start': str(dataset_start),
        'dataset_end': str(dataset_end),
        'duration': str(duration),
        'processing_time_seconds': elapsed_total
    }

    with open(f'{output_dir}/metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)

    return graphs_created, graphs_with_attacks

# ============================================================================
# EJECUTAR VERSIÓN OPTIMIZADA
# ============================================================================

import gc
gc.collect()

print("\n🚀 VERSIÓN OPTIMIZADA - MUCHO MÁS RÁPIDA\n")

num_graphs, num_attacks = create_graphs_OPTIMIZED(
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl',
    timestamp_col_start,
    src_ip_col,
    dst_ip_col,
    label_col,
    window_minutes=10,
    output_dir='/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_OPTIMIZED'
)

print(f"\n🎉 ¡COMPLETADO!")
print(f"   Total grafos: {num_graphs}")
print(f"   Con ataques: {num_attacks}")


🚀 VERSIÓN OPTIMIZADA - MUCHO MÁS RÁPIDA

 ⚡ VERSIÓN ULTRA-OPTIMIZADA

✅ Cargando mapeo de IPs...
   Nodos: 205,801

⚡ PASADA ÚNICA: Procesando chunks...

  Chunk 1/81 | Flows: 250,000 | Ventanas en buffer: 9 | Tiempo: 1.1s
  Chunk 2/81 | Flows: 500,000 | Ventanas en buffer: 14 | Tiempo: 2.1s
  Chunk 3/81 | Flows: 750,000 | Ventanas en buffer: 18 | Tiempo: 3.0s
  Chunk 4/81 | Flows: 1,000,000 | Ventanas en buffer: 21 | Tiempo: 3.7s
  Chunk 5/81 | Flows: 1,250,000 | Ventanas en buffer: 30 | Tiempo: 4.5s
  Chunk 6/81 | Flows: 1,500,000 | Ventanas en buffer: 36 | Tiempo: 5.8s
  Chunk 7/81 | Flows: 1,750,000 | Ventanas en buffer: 41 | Tiempo: 7.2s
  Chunk 8/81 | Flows: 2,000,000 | Ventanas en buffer: 49 | Tiempo: 9.4s
  Chunk 9/81 | Flows: 2,250,000 | Ventanas en buffer: 77 | Tiempo: 11.3s
  Chunk 10/81 | Flows: 2,500,000 | Ventanas en buffer: 83 | Tiempo: 12.4s
  Chunk 11/81 | Flows: 2,750,000 | Ventanas en buffer: 90 | Tiempo: 13.8s
  Chunk 12/81 | Flows: 3,000,000 | Ventanas en buffer: 

In [ ]:
# ============================================================================
# VERIFICACIÓN EXHAUSTIVA DE GRAFOS
# ============================================================================

import pickle
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

timestamp_col_start = 'FLOW_START_MILLISECONDS'
timestamp_col_end = 'FLOW_END_MILLISECONDS'
src_ip_col = 'IPV4_SRC_ADDR'
dst_ip_col = 'IPV4_DST_ADDR'
label_col = 'Label'

def read_graphs_generator(graphs_file):
    """Lee grafos de forma incremental"""
    with open(graphs_file, 'rb') as f:
        while True:
            try:
                yield pickle.load(f)
            except EOFError:
                break

# ============================================================================
# 1. VERIFICACIÓN DE IDs DE NODOS (CONSISTENCIA GLOBAL)
# ============================================================================

def verify_node_ids(graphs_file, mapping_file, num_graphs_to_check=10):
    """
    Verifica que los IDs de nodos sean consistentes entre grafos
    """

    print("="*70)
    print("1️⃣  VERIFICACIÓN DE IDs DE NODOS")
    print("="*70)

    # Cargar mapeo global
    with open(mapping_file, 'rb') as f:
        global_ip_to_idx = pickle.load(f)

    total_nodes = len(global_ip_to_idx)
    print(f"\n📊 Mapeo global: {total_nodes:,} nodos\n")

    # Verificar en múltiples grafos
    for i, graph in enumerate(read_graphs_generator(graphs_file)):

        if i >= num_graphs_to_check:
            break

        print(f"Grafo {i} (Ventana {graph.window_id}):")
        print(f"  num_nodes (declarado): {graph.num_nodes:,}")
        print(f"  num_nodes (esperado):  {total_nodes:,}")

        # Verificar que coincida
        if graph.num_nodes == total_nodes:
            print(f"  ✅ Correcto")
        else:
            print(f"  ❌ ERROR: No coincide")

        # Verificar edge_index
        max_node_id = graph.edge_index.max().item()
        min_node_id = graph.edge_index.min().item()

        print(f"  Rango de IDs en aristas: [{min_node_id}, {max_node_id}]")

        if max_node_id < total_nodes:
            print(f"  ✅ Todos los IDs dentro del rango")
        else:
            print(f"  ❌ ERROR: ID fuera de rango ({max_node_id} >= {total_nodes})")

        # Verificar nodos activos
        unique_nodes = torch.unique(graph.edge_index).numel()
        print(f"  Nodos activos (con aristas): {unique_nodes:,}")
        print(f"  Nodos inactivos: {total_nodes - unique_nodes:,}")
        print()

# ============================================================================
# 2. ESTADÍSTICAS DE GRAFOS
# ============================================================================

def analyze_graph_statistics(graphs_file):
    """
    Estadísticas detalladas de todos los grafos
    """

    print("="*70)
    print("2️⃣  ESTADÍSTICAS DE GRAFOS")
    print("="*70)

    stats = {
        'num_nodes': [],
        'num_edges': [],
        'num_active_nodes': [],
        'num_flows': [],
        'label': [],
        'attack_flows': [],
        'attack_ratio': []
    }

    for graph in read_graphs_generator(graphs_file):
        stats['num_nodes'].append(graph.num_nodes)
        stats['num_edges'].append(graph.edge_index.shape[1])
        stats['num_active_nodes'].append(getattr(graph, 'num_active_nodes', 0))
        stats['num_flows'].append(getattr(graph, 'num_flows_original', 0))
        stats['label'].append(graph.y.item())
        stats['attack_flows'].append(getattr(graph, 'attack_flows', 0))
        stats['attack_ratio'].append(getattr(graph, 'attack_ratio', 0.0))

    df_stats = pd.DataFrame(stats)

    print(f"\n📊 Resumen general:")
    print(f"  Total grafos: {len(df_stats)}")
    print(f"  Normal (0): {(df_stats['label']==0).sum()} ({(df_stats['label']==0).sum()/len(df_stats)*100:.1f}%)")
    print(f"  Ataque (1): {(df_stats['label']==1).sum()} ({(df_stats['label']==1).sum()/len(df_stats)*100:.1f}%)")

    print(f"\n📊 Estadísticas de aristas:")
    print(f"  Min:    {df_stats['num_edges'].min():,}")
    print(f"  Max:    {df_stats['num_edges'].max():,}")
    print(f"  Media:  {df_stats['num_edges'].mean():,.0f}")
    print(f"  Mediana: {df_stats['num_edges'].median():,.0f}")

    print(f"\n📊 Estadísticas de nodos activos:")
    print(f"  Min:    {df_stats['num_active_nodes'].min():,}")
    print(f"  Max:    {df_stats['num_active_nodes'].max():,}")
    print(f"  Media:  {df_stats['num_active_nodes'].mean():,.0f}")

    print(f"\n📊 Estadísticas de flows:")
    print(f"  Min:    {df_stats['num_flows'].min():,}")
    print(f"  Max:    {df_stats['num_flows'].max():,}")
    print(f"  Media:  {df_stats['num_flows'].mean():,.0f}")

    print(f"\n📊 Grafos con ataques:")
    attack_graphs = df_stats[df_stats['label'] == 1]
    if len(attack_graphs) > 0:
        print(f"  Flows de ataque promedio: {attack_graphs['attack_flows'].mean():,.0f}")
        print(f"  Ratio de ataque promedio: {attack_graphs['attack_ratio'].mean()*100:.1f}%")
        print(f"  Min ratio: {attack_graphs['attack_ratio'].min()*100:.1f}%")
        print(f"  Max ratio: {attack_graphs['attack_ratio'].max()*100:.1f}%")

    return df_stats

# ============================================================================
# 3. VISUALIZACIÓN DE GRAFOS
# ============================================================================

def visualize_sample_graphs(graphs_file, num_samples=3):
    """
    Visualiza grafos de ejemplo
    """

    print("\n" + "="*70)
    print("3️⃣  VISUALIZACIÓN DE GRAFOS DE EJEMPLO")
    print("="*70)

    # Seleccionar grafos: 1 normal, 1 ataque bajo, 1 ataque alto
    graphs_to_viz = []

    for graph in read_graphs_generator(graphs_file):

        # Grafo normal
        if graph.y.item() == 0 and len(graphs_to_viz) == 0:
            graphs_to_viz.append(('Normal', graph))

        # Grafo con ataque (ratio bajo)
        if graph.y.item() == 1 and len(graphs_to_viz) == 1:
            if graph.attack_ratio < 0.1:
                graphs_to_viz.append(('Ataque (ratio bajo)', graph))

        # Grafo con ataque (ratio alto)
        if graph.y.item() == 1 and len(graphs_to_viz) == 2:
            if graph.attack_ratio > 0.5:
                graphs_to_viz.append(('Ataque (ratio alto)', graph))
                break

    # Visualizar
    for label, graph in graphs_to_viz:
        print(f"\n📊 {label} (Ventana {graph.window_id}):")
        print(f"  Label: {graph.y.item()}")
        print(f"  Nodos totales: {graph.num_nodes:,}")
        print(f"  Nodos activos: {graph.num_active_nodes:,}")
        print(f"  Aristas: {graph.edge_index.shape[1]:,}")
        print(f"  Flows originales: {graph.num_flows_original:,}")

        if graph.y.item() == 1:
            print(f"  Flows de ataque: {graph.attack_flows:,} ({graph.attack_ratio*100:.1f}%)")

        # Features
        print(f"  Node features shape: {graph.x.shape}")
        print(f"  Edge features shape: {graph.edge_attr.shape}")

        # Sample de node features
        print(f"\n  Sample de node features (primeros 5 nodos activos):")
        active_mask = graph.x[:, 0] > 0  # Nodos con flows > 0
        active_indices = torch.where(active_mask)[0][:5]

        for idx in active_indices:
            features = graph.x[idx].tolist()
            print(f"    Nodo {idx.item()}: {[f'{f:.2f}' for f in features]}")

# ============================================================================
# 4. VERIFICAR INFORMACIÓN DE ATAQUES (CRÍTICO)
# ============================================================================

def verify_attack_information(graphs_file, filtered_chunks_file,
                             timestamp_col, src_col, dst_col, label_col,
                             num_windows_to_check=5):
    """
    Verifica que la información de ataques sea correcta
    comparando con los chunks originales
    """

    print("\n" + "="*70)
    print("4️⃣  VERIFICACIÓN DE INFORMACIÓN DE ATAQUES")
    print("="*70)

    # Cargar mapeo de IPs
    with open('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_fixed_nodes_label0/ip_to_idx_mapping.pkl', 'rb') as f:
        global_ip_to_idx = pickle.load(f)

    # Obtener dataset_start
    for chunk in read_filtered_chunks(filtered_chunks_file):
        ns_vals = chunk[timestamp_col].values.astype('int64')
        chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')
        dataset_start = chunk[timestamp_col].min()
        del chunk
        break

    window_size = pd.Timedelta(minutes=10)

    print(f"\n🔍 Verificando {num_windows_to_check} ventanas...\n")

    # Seleccionar grafos con ataque
    attack_graphs = []
    for graph in read_graphs_generator(graphs_file):
        if graph.y.item() == 1:
            attack_graphs.append(graph)
            if len(attack_graphs) >= num_windows_to_check:
                break

    for graph in attack_graphs:
        window_id = graph.window_id
        window_start = dataset_start + window_id * window_size
        window_end = window_start + window_size

        print(f"Ventana {window_id} ({window_start} → {window_end}):")
        print(f"  Label del grafo: {graph.y.item()}")
        print(f"  Flows de ataque (según grafo): {graph.attack_flows:,}")
        print(f"  Ratio de ataque (según grafo): {graph.attack_ratio*100:.1f}%")

        # Reconstruir ventana desde chunks para verificar
        window_flows = []
        for chunk in read_filtered_chunks(filtered_chunks_file):
            ns_vals = chunk[timestamp_col].values.astype('int64')
            chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')

            mask = (
                (chunk[timestamp_col] >= window_start) &
                (chunk[timestamp_col] < window_end)
            )

            chunk_window = chunk[mask]
            if len(chunk_window) > 0:
                window_flows.append(chunk_window)

            del chunk

        if len(window_flows) > 0:
            window_df = pd.concat(window_flows, ignore_index=True)

            # Contar ataques reales
            actual_attacks = (window_df[label_col] != 0).sum()
            actual_ratio = actual_attacks / len(window_df)

            print(f"  Flows de ataque (recalculado): {actual_attacks:,}")
            print(f"  Ratio de ataque (recalculado): {actual_ratio*100:.1f}%")

            # Verificar
            if graph.attack_flows == actual_attacks:
                print(f"  ✅ Coincide")
            else:
                print(f"  ⚠️  DIFERENCIA: {abs(graph.attack_flows - actual_attacks)} flows")

            # Mostrar tipos de ataque
            if actual_attacks > 0:
                attack_types = window_df[window_df[label_col] != 0][label_col].unique()
                print(f"  Tipos de ataque presentes: {list(attack_types)}")

            del window_df

        print()

# ============================================================================
# 5. VERIFICAR QUE NO TENEMOS INFO DE LABELS EN LOS GRAFOS
# ============================================================================

def verify_no_label_leakage(graphs_file):
    """
    Verifica que los grafos NO contengan información de qué flows son ataques
    (solo deben tener el label global 0/1)
    """

    print("\n" + "="*70)
    print("5️⃣  VERIFICACIÓN DE NO-FILTRACIÓN DE LABELS")
    print("="*70)

    print("\n🔍 Verificando que los grafos NO contengan info de flows individuales...\n")

    for i, graph in enumerate(read_graphs_generator(graphs_file)):

        if i >= 10:
            break

        # Atributos del grafo
        graph_attrs = dir(graph)

        print(f"Grafo {i} (Ventana {graph.window_id}, Label={graph.y.item()}):")

        # Atributos estándar esperados
        expected_attrs = ['x', 'edge_index', 'edge_attr', 'y', 'num_nodes']

        # Atributos de metadata (permitidos)
        metadata_attrs = ['window_id', 'window_start', 'window_end',
                         'num_active_nodes', 'num_flows_original',
                         'attack_flows', 'attack_ratio', 'num_edges_actual']

        # Buscar atributos sospechosos
        suspicious = []
        for attr in graph_attrs:
            if attr.startswith('_'):
                continue

            if attr not in expected_attrs and attr not in metadata_attrs:
                suspicious.append(attr)

        if len(suspicious) > 0:
            print(f"  ⚠️  Atributos inesperados: {suspicious}")
        else:
            print(f"  ✅ Solo atributos esperados")

        # Verificar que NO hay labels por flow/edge
        if hasattr(graph, 'edge_labels'):
            print(f"  ❌ ERROR: Tiene 'edge_labels' (filtración de info)")

        if hasattr(graph, 'node_labels'):
            print(f"  ❌ ERROR: Tiene 'node_labels' (filtración de info)")

        # Verificar dimensiones
        print(f"  Shape de x: {graph.x.shape}")
        print(f"  Shape de edge_attr: {graph.edge_attr.shape}")
        print(f"  Shape de y: {graph.y.shape}")
        print()

# ============================================================================
# 6. DISTRIBUCIÓN TEMPORAL
# ============================================================================

def analyze_temporal_distribution(graphs_file):
    """
    Analiza la distribución temporal de ataques
    """

    print("\n" + "="*70)
    print("6️⃣  DISTRIBUCIÓN TEMPORAL DE ATAQUES")
    print("="*70)

    window_ids = []
    labels = []
    attack_ratios = []

    for graph in read_graphs_generator(graphs_file):
        window_ids.append(graph.window_id)
        labels.append(graph.y.item())
        attack_ratios.append(graph.attack_ratio if hasattr(graph, 'attack_ratio') else 0.0)

    df = pd.DataFrame({
        'window_id': window_ids,
        'label': labels,
        'attack_ratio': attack_ratios
    })

    print(f"\n📊 Distribución por bloques de 100 ventanas:")

    df['block'] = (df['window_id'] // 100) * 100

    summary = df.groupby('block').agg({
        'label': ['count', 'sum'],
        'attack_ratio': 'mean'
    })

    print(f"\n{'Ventanas':<15} {'Total':<10} {'Ataques':<10} {'% Ataque':<12} {'Ratio Avg':<12}")
    print("-" * 65)

    for block, row in summary.iterrows():
        total = int(row['label']['count'])
        attacks = int(row['label']['sum'])
        pct = attacks / total * 100 if total > 0 else 0
        ratio_avg = row['attack_ratio']['mean'] * 100

        print(f"{block:>4}-{block+99:<7} {total:<10} {attacks:<10} {pct:>5.1f}%       {ratio_avg:>5.1f}%")

    return df

# ============================================================================
# EJECUTAR TODAS LAS VERIFICACIONES
# ============================================================================

graphs_file = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_OPTIMIZED/graphs_optimized.pkl'
mapping_file = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_fixed_nodes_label0/ip_to_idx_mapping.pkl'
filtered_chunks_file = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl'

print("\n" + "="*70)
print("🔍 VERIFICACIÓN EXHAUSTIVA DE GRAFOS")
print("="*70)

# 1. Verificar IDs de nodos
verify_node_ids(graphs_file, mapping_file, num_graphs_to_check=10)

# 2. Estadísticas generales
df_stats = analyze_graph_statistics(graphs_file)

# 3. Visualizar grafos de ejemplo
visualize_sample_graphs(graphs_file)

# 4. Verificar información de ataques
verify_attack_information(
    graphs_file,
    filtered_chunks_file,
    timestamp_col_start,
    src_ip_col,
    dst_ip_col,
    label_col,
    num_windows_to_check=5
)

# 5. Verificar no-filtración de labels
verify_no_label_leakage(graphs_file)

# 6. Distribución temporal
df_temporal = analyze_temporal_distribution(graphs_file)

print("\n" + "="*70)
print("✅ VERIFICACIÓN COMPLETADA")
print("="*70)


🔍 VERIFICACIÓN EXHAUSTIVA DE GRAFOS
1️⃣  VERIFICACIÓN DE IDs DE NODOS

📊 Mapeo global: 205,801 nodos

Grafo 0 (Ventana 0):
  num_nodes (declarado): 205,801
  num_nodes (esperado):  205,801
  ✅ Correcto
  Rango de IDs en aristas: [0, 205525]
  ✅ Todos los IDs dentro del rango
  Nodos activos (con aristas): 2,351
  Nodos inactivos: 203,450

Grafo 1 (Ventana 1):
  num_nodes (declarado): 205,801
  num_nodes (esperado):  205,801
  ✅ Correcto
  Rango de IDs en aristas: [0, 205602]
  ✅ Todos los IDs dentro del rango
  Nodos activos (con aristas): 2,648
  Nodos inactivos: 203,153

Grafo 2 (Ventana 2):
  num_nodes (declarado): 205,801
  num_nodes (esperado):  205,801
  ✅ Correcto
  Rango de IDs en aristas: [0, 205721]
  ✅ Todos los IDs dentro del rango
  Nodos activos (con aristas): 2,485
  Nodos inactivos: 203,316

Grafo 3 (Ventana 3):
  num_nodes (declarado): 205,801
  num_nodes (esperado):  205,801
  ✅ Correcto
  Rango de IDs en aristas: [0, 205523]
  ✅ Todos los IDs dentro del rango
  Nodo

In [ ]:
# ============================================================================
# Verificar que los IDs de nodos son consistentes entre grafos
# ============================================================================

def verify_node_consistency(graphs_file, mapping_file, filtered_chunks_file,
                           src_col, dst_col, timestamp_col):
    """
    Verifica que el nodo X siempre representa la misma IP
    """

    import pandas as pd

    # Cargar mapeo
    with open(mapping_file, 'rb') as f:
        global_ip_to_idx = pickle.load(f)

    idx_to_ip = {v: k for k, v in global_ip_to_idx.items()}

    print("="*70)
    print("🔍 VERIFICACIÓN DE CONSISTENCIA DE NODOS")
    print("="*70)

    # Obtener dataset_start
    for chunk in read_filtered_chunks(filtered_chunks_file):
        ns_vals = chunk[timestamp_col].values.astype('int64')
        chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')
        dataset_start = chunk[timestamp_col].min()
        del chunk
        break

    window_size = pd.Timedelta(minutes=10)

    # Seleccionar nodos para verificar
    test_node_ids = [0, 1, 2, 57, 131]

    print(f"\n📊 Mapeo global de nodos de prueba:")
    for node_id in test_node_ids:
        print(f"  Nodo {node_id}: IP = {idx_to_ip[node_id]}")

    # Verificar en 3 grafos diferentes
    graphs_to_check = [0, 149, 319]

    for i, graph in enumerate(read_graphs_generator(graphs_file)):

        if graph.window_id not in graphs_to_check:
            continue

        window_id = graph.window_id
        window_start = dataset_start + window_id * window_size
        window_end = window_start + window_size

        print(f"\n{'='*70}")
        print(f"Grafo {i} (Ventana {window_id}: {window_start})")
        print(f"{'='*70}")

        # Reconstruir ventana desde chunks
        window_flows = []
        for chunk in read_filtered_chunks(filtered_chunks_file):
            ns_vals = chunk[timestamp_col].values.astype('int64')
            chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')

            mask = (
                (chunk[timestamp_col] >= window_start) &
                (chunk[timestamp_col] < window_end)
            )

            chunk_window = chunk[mask]
            if len(chunk_window) > 0:
                window_flows.append(chunk_window)

            del chunk

        window_df = pd.concat(window_flows, ignore_index=True)

        # Verificar nodos de prueba
        for node_id in test_node_ids:
            expected_ip = idx_to_ip[node_id]

            # Contar flows de esta IP en la ventana
            as_src = (window_df[src_col] == expected_ip).sum()
            as_dst = (window_df[dst_col] == expected_ip).sum()
            total = as_src + as_dst

            # Features del grafo
            node_features = graph.x[node_id].tolist()
            graph_total_flows = node_features[0]
            graph_as_src = node_features[1]
            graph_as_dst = node_features[2]

            print(f"\nNodo {node_id} (IP: {expected_ip}):")
            print(f"  Flows en ventana real:     Total={total}, Src={as_src}, Dst={as_dst}")
            print(f"  Flows en grafo (features): Total={graph_total_flows:.0f}, Src={graph_as_src:.0f}, Dst={graph_as_dst:.0f}")

            # Verificar
            if total == graph_total_flows:
                print(f"  ✅ COINCIDE - El nodo {node_id} representa correctamente la IP {expected_ip}")
            else:
                print(f"  ⚠️  DIFERENCIA de {abs(total - graph_total_flows):.0f} flows")

        del window_df

        if len([g for g in read_graphs_generator(graphs_file) if g.window_id in graphs_to_check]) >= len(graphs_to_check):
            break

# Ejecutar verificación
verify_node_consistency(
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_OPTIMIZED/graphs_optimized.pkl',
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_fixed_nodes_label0/ip_to_idx_mapping.pkl',
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl',
    src_ip_col,
    dst_ip_col,
    timestamp_col_start
)


🔍 VERIFICACIÓN DE CONSISTENCIA DE NODOS

📊 Mapeo global de nodos de prueba:
  Nodo 0: IP = 0.0.0.0
  Nodo 1: IP = 1.0.0.3
  Nodo 2: IP = 1.0.145.95
  Nodo 57: IP = 1.10.209.197
  Nodo 131: IP = 1.161.211.39

Grafo 0 (Ventana 0: 2018-02-14 12:28:07.704999936)

Nodo 0 (IP: 0.0.0.0):
  Flows en ventana real:     Total=478, Src=239, Dst=239
  Flows en grafo (features): Total=478, Src=239, Dst=239
  ✅ COINCIDE - El nodo 0 representa correctamente la IP 0.0.0.0

Nodo 1 (IP: 1.0.0.3):
  Flows en ventana real:     Total=0, Src=0, Dst=0
  Flows en grafo (features): Total=0, Src=0, Dst=0
  ✅ COINCIDE - El nodo 1 representa correctamente la IP 1.0.0.3

Nodo 2 (IP: 1.0.145.95):
  Flows en ventana real:     Total=0, Src=0, Dst=0
  Flows en grafo (features): Total=0, Src=0, Dst=0
  ✅ COINCIDE - El nodo 2 representa correctamente la IP 1.0.145.95

Nodo 57 (IP: 1.10.209.197):
  Flows en ventana real:     Total=3, Src=3, Dst=0
  Flows en grafo (features): Total=3, Src=3, Dst=0
  ✅ COINCIDE - El nodo 57